# RoadSafe India — Master State/UT Analytical Dataset

## Objective

This notebook combines the cleaned accident, fatality and population datasets
into a single State/UT-level analytical dataset.

The master dataset will contain:

- Accident counts for 2020–2024.
- Fatality counts for 2020–2024.
- 2024 projected population.
- Accident rates per 100,000 projected population.
- Fatality rates per 100,000 projected population.
- Fatalities per 100 reported accidents.
- Five-year average accident burden.
- Five-year average fatality burden.
- Accident and fatality changes between 2020 and 2024.

The master dataset will serve as the primary analytical table for subsequent
statistical analysis and dashboard development.

In [30]:
import pandas as pd

In [31]:
# Load cleaned accident data

accident_path = (
    "../data/processed/"
    "state_wise_road_accidents_2020_2024_cleaned.csv"
)

accident_df = pd.read_csv(accident_path)

# Load cleaned fatality data

fatality_path = (
    "../data/processed/"
    "state_wise_road_fatalities_2020_2024_cleaned.csv"
)

fatality_df = pd.read_csv(fatality_path)

# Load 2024 population data

population_path = (
    "../data/processed/"
    "state_population_2024.csv"
)

population_df = pd.read_csv(population_path)


print("Accident dataset:", accident_df.shape)
print("Fatality dataset:", fatality_df.shape)
print("Population dataset:", population_df.shape)

Accident dataset: (39, 14)
Fatality dataset: (36, 13)
Population dataset: (37, 3)


In [32]:
accident_columns = [
    "2020_accidents",
    "2021_accidents",
    "2022_accidents",
    "2023_accidents",
    "2024_accidents"
]

fatality_columns = [
    "2020_killed",
    "2021_killed",
    "2022_killed",
    "2023_killed",
    "2024_killed"
]

years = [2020, 2021, 2022, 2023, 2024]

print("Accident columns:")
print(accident_columns)

print("\nFatality columns:")
print(fatality_columns)

Accident columns:
['2020_accidents', '2021_accidents', '2022_accidents', '2023_accidents', '2024_accidents']

Fatality columns:
['2020_killed', '2021_killed', '2022_killed', '2023_killed', '2024_killed']


In [33]:
accident_master = accident_df[
    ["state"] + accident_columns
].copy()

# Remove rows without State/UT information
accident_master = accident_master[
    accident_master["state"].notna()
].copy()

# Remove aggregate rows such as Total (All India)
accident_master = accident_master[
    ~accident_master["state"].str.contains(
        "total|india",
        case=False,
        na=False
    )
].copy()

print("Accident records:")
print(accident_master.shape)

display(accident_master.head())

Accident records:
(36, 6)


,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0


In [34]:
fatality_master = fatality_df[
    ["state"] + fatality_columns
].copy()

# Remove rows without State/UT information
fatality_master = fatality_master[
    fatality_master["state"].notna()
].copy()

# Remove aggregate rows if present
fatality_master = fatality_master[
    ~fatality_master["state"].str.contains(
        "total|india",
        case=False,
        na=False
    )
].copy()

print("Fatality records:")
print(fatality_master.shape)

display(fatality_master.head())

Fatality records:
(36, 6)


,state,2020_killed,2021_killed,2022_killed,2023_killed,2024_killed
0,Andhra Pradesh,7039.0,8186,8293,8137,8346
1,Arunachal Pradesh,73.0,157,148,145,168
2,Assam,2629.0,3036,2994,3296,3351
3,Bihar,6699.0,7660,8898,8873,9347
4,Chhattisgarh,4606.0,5371,5834,6166,6945


In [35]:
population_master = population_df[
    ["state", "population_2024"]
].copy()

print("Population records:")
print(population_master.shape)

display(population_master.head())

Population records:
(37, 2)


,state,population_2024
0,Andhra Pradesh,53340000
1,Arunachal Pradesh,1576000
2,Assam,36047000
3,Bihar,128592000
4,Chhattisgarh,30524000


In [36]:
# State/UT harmonization and compatibility check

# Basic text cleaning
for dataframe in [
    accident_master,
    fatality_master,
    population_master
]:
    dataframe["state"] = (
        dataframe["state"]
        .astype("string")
        .str.strip()
    )

# Harmonize names in accident data

accident_master["state"] = accident_master["state"].replace({
    "Delhi": "N.C.T of Delhi",
    "J & K #": "Jammu & Kashmir"
})

# Harmonize names in fatality data

fatality_master["state"] = fatality_master["state"].replace({
    "Delhi": "N.C.T of Delhi",
    "J & K #": "Jammu & Kashmir"
})

# Combine Dadra & Nagar Haveli + Daman & Diu

separate_ut_names = [
    "Dadra & Nagar Haveli",
    "Daman & Diu"
]

# Find and combine their population
combined_population = population_master.loc[
    population_master["state"].isin(separate_ut_names),
    "population_2024"
].sum()

print(
    "Combined population of Dadra & Nagar Haveli "
    f"and Daman & Diu: {combined_population:,.0f}"
)

# Remove the two separate records
population_master = population_master[
    ~population_master["state"].isin(separate_ut_names)
].copy()

# Remove a combined record too, if one already exists
population_master = population_master[
    population_master["state"]
    != "Dadra & Nagar Haveli and Daman & Diu"
].copy()

# Add exactly one combined record
combined_ut = pd.DataFrame({
    "state": [
        "Dadra & Nagar Haveli and Daman & Diu"
    ],
    "population_2024": [
        combined_population
    ]
})

population_master = pd.concat(
    [population_master, combined_ut],
    ignore_index=True
)


# Create State/UT sets

accident_states = set(
    accident_master["state"].dropna()
)

fatality_states = set(
    fatality_master["state"].dropna()
)

population_states = set(
    population_master["state"].dropna()
)

# Compare every dataset against every other dataset

print("\n========== STATE/UT COMPATIBILITY ==========")

print("\nAccident vs Fatality")
print("Only in accident:")
print(sorted(accident_states - fatality_states))
print("Only in fatality:")
print(sorted(fatality_states - accident_states))

print("\nAccident vs Population")
print("Only in accident:")
print(sorted(accident_states - population_states))
print("Only in population:")
print(sorted(population_states - accident_states))

print("\nFatality vs Population")
print("Only in fatality:")
print(sorted(fatality_states - population_states))
print("Only in population:")
print(sorted(population_states - fatality_states))

# Check row counts and uniqueness

print("\n========== DATASET COUNTS ==========")

print(
    "Accident rows:",
    len(accident_master),
    "| Unique:",
    accident_master["state"].nunique()
)

print(
    "Fatality rows:",
    len(fatality_master),
    "| Unique:",
    fatality_master["state"].nunique()
)

print(
    "Population rows:",
    len(population_master),
    "| Unique:",
    population_master["state"].nunique()
)

# Final safety checks

if (
    accident_master["state"].duplicated().any()
    or fatality_master["state"].duplicated().any()
    or population_master["state"].duplicated().any()
):
    raise ValueError(
        "Duplicate State/UT found in at least one dataset."
    )

if (
    accident_states != fatality_states
    or accident_states != population_states
):
    raise ValueError(
        "State/UT coverage is still not identical. "
        "Review the mismatch lists above."
    )

print("\nAll three datasets have matching State/UT coverage.")
print(" No duplicate State/UT keys found.")
print(" Safe to merge.")

Combined population of Dadra & Nagar Haveli and Daman & Diu: 1,356,000

========== STATE/UT COMPATIBILITY ==========

Accident vs Fatality
Only in accident:
[]
Only in fatality:
[]

Accident vs Population
Only in accident:
[]
Only in population:
[]

Fatality vs Population
Only in fatality:
[]
Only in population:
[]

========== DATASET COUNTS ==========
Accident rows: 36 | Unique: 36
Fatality rows: 36 | Unique: 36
Population rows: 36 | Unique: 36

All three datasets have matching State/UT coverage.
 No duplicate State/UT keys found.
 Safe to merge.


In [37]:
master_df = pd.merge(
    accident_master,
    fatality_master,
    on="state",
    how="inner",
    validate="one_to_one"
)

print("After accident + fatality merge:")
print(master_df.shape)

display(master_df.head())

After accident + fatality merge:
(36, 11)


,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,2020_killed,2021_killed,2022_killed,2023_killed,2024_killed
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0,7039.0,8186,8293,8137,8346
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,73.0,157,148,145,168
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0,2629.0,3036,2994,3296,3351
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0,6699.0,7660,8898,8873,9347
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0,4606.0,5371,5834,6166,6945


In [38]:
master_df = pd.merge(
    master_df,
    population_master,
    on="state",
    how="inner",
    validate="one_to_one"
)

print("Final master dataset shape:")
print(master_df.shape)

display(master_df.head())

Final master dataset shape:
(36, 12)


,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,2020_killed,2021_killed,2022_killed,2023_killed,2024_killed,population_2024
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0,7039.0,8186,8293,8137,8346,53340000
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,73.0,157,148,145,168,1576000
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0,2629.0,3036,2994,3296,3351,36047000
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0,6699.0,7660,8898,8873,9347,128592000
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0,4606.0,5371,5834,6166,6945,30524000


In [39]:
master_df["five_year_average_accidents"] = (
    master_df[accident_columns]
    .mean(axis=1)
)

display(
    master_df[
        [
            "state",
            *accident_columns,
            "five_year_average_accidents"
        ]
    ].head()
)

,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,five_year_average_accidents
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0,20364.0
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,241.6
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0,7259.6
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0,10323.4
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0,13127.0


In [40]:
master_df["fatality_years_available"] = (
    master_df[fatality_columns]
    .notna()
    .sum(axis=1)
)

master_df["five_year_average_fatalities"] = (
    master_df[fatality_columns]
    .mean(axis=1)
)

display(
    master_df[
        [
            "state",
            *fatality_columns,
            "fatality_years_available",
            "five_year_average_fatalities"
        ]
    ].head()
)

,state,2020_killed,2021_killed,2022_killed,2023_killed,2024_killed,fatality_years_available,five_year_average_fatalities
0,Andhra Pradesh,7039.0,8186,8293,8137,8346,5,8000.2
1,Arunachal Pradesh,73.0,157,148,145,168,5,138.2
2,Assam,2629.0,3036,2994,3296,3351,5,3061.2
3,Bihar,6699.0,7660,8898,8873,9347,5,8295.4
4,Chhattisgarh,4606.0,5371,5834,6166,6945,5,5784.4


In [41]:
# Accident Change from 2020 to 2024
master_df["accident_change_2020_to_2024"] = (
    master_df["2024_accidents"]
    - master_df["2020_accidents"]
)

master_df["accident_percent_change_2020_to_2024"] = (
    master_df["accident_change_2020_to_2024"]
    / master_df["2020_accidents"]
    * 100
)

In [42]:
# Fatality Change from 2020 to 2024
master_df["fatality_change_2020_to_2024"] = (
    master_df["2024_killed"]
    - master_df["2020_killed"]
)

master_df["fatality_percent_change_2020_to_2024"] = (
    master_df["fatality_change_2020_to_2024"]
    / master_df["2020_killed"]
    * 100
)

In [43]:
master_df["accidents_per_100k_population"] = (
    master_df["2024_accidents"]
    / master_df["population_2024"]
    * 100_000
)

In [44]:
master_df["fatalities_per_100k_population"] = (
    master_df["2024_killed"]
    / master_df["population_2024"]
    * 100_000
)

In [45]:
master_df["fatalities_per_100_accidents"] = (
    master_df["2024_killed"]
    / master_df["2024_accidents"]
    * 100
)

In [46]:
master_df["accident_rank_2024"] = (
    master_df["2024_accidents"]
    .rank(
        ascending=False,
        method="min"
    )
)

master_df["fatality_rank_2024"] = (
    master_df["2024_killed"]
    .rank(
        ascending=False,
        method="min"
    )
)

master_df["accident_rate_rank_2024"] = (
    master_df["accidents_per_100k_population"]
    .rank(
        ascending=False,
        method="min"
    )
)

master_df["fatality_rate_rank_2024"] = (
    master_df["fatalities_per_100k_population"]
    .rank(
        ascending=False,
        method="min"
    )
)

In [47]:
master_df["accident_vs_fatality_rank_difference"] = (
    master_df["accident_rank_2024"]
    - master_df["fatality_rank_2024"]
)

display(
    master_df[
        [
            "state",
            "accident_rank_2024",
            "fatality_rank_2024",
            "accident_rate_rank_2024",
            "fatality_rate_rank_2024"
        ]
    ].sort_values("accident_rank_2024")
)

,state,accident_rank_2024,fatality_rank_2024,accident_rate_rank_2024,fatality_rate_rank_2024
22,Tamil Nadu,1.0,2.0,3.0,1.0
12,Madhya Pradesh,2.0,4.0,7.0,7.0
11,Kerala,3.0,17.0,2.0,16.0
26,Uttar Pradesh,4.0,1.0,23.0,20.0
10,Karnataka,5.0,5.0,8.0,5.0
13,Maharashtra,6.0,3.0,16.0,14.0
23,Telangana,7.0,9.0,6.0,3.0
20,Rajasthan,8.0,6.0,14.0,11.0
0,Andhra Pradesh,9.0,8.0,11.0,8.0
6,Gujarat,10.0,10.0,20.0,17.0


In [48]:
master_columns = [
    "state",

    # Accident counts
    "2020_accidents",
    "2021_accidents",
    "2022_accidents",
    "2023_accidents",
    "2024_accidents",

    # Fatality counts
    "2020_killed",
    "2021_killed",
    "2022_killed",
    "2023_killed",
    "2024_killed",

    # Population
    "population_2024",

    # 2024 normalized metrics
    "accidents_per_100k_population",
    "fatalities_per_100k_population",
    "fatalities_per_100_accidents",

    # Five-year averages
    "five_year_average_accidents",
    "five_year_average_fatalities",
    "fatality_years_available",

    # Changes
    "accident_change_2020_to_2024",
    "accident_percent_change_2020_to_2024",
    "fatality_change_2020_to_2024",
    "fatality_percent_change_2020_to_2024",

    # Rankings
    "accident_rank_2024",
    "fatality_rank_2024",
    "accident_rate_rank_2024",
    "fatality_rate_rank_2024"
]

master_df = master_df[master_columns]

display(master_df.head())

,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,2020_killed,2021_killed,2022_killed,2023_killed,...,five_year_average_fatalities,fatality_years_available,accident_change_2020_to_2024,accident_percent_change_2020_to_2024,fatality_change_2020_to_2024,fatality_percent_change_2020_to_2024,accident_rank_2024,fatality_rank_2024,accident_rate_rank_2024,fatality_rate_rank_2024
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0,7039.0,8186,8293,8137,...,8000.2,5,48.0,0.246040,1307.0,18.567978,9.0,8.0,11.0,8.0
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,73.0,157,148,145,...,138.2,5,143.0,106.716418,95.0,130.136986,27.0,27.0,24.0,18.0
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0,2629.0,3036,2994,3296,...,3061.2,5,1253.0,18.999242,722.0,27.462914,16.0,18.0,19.0,22.0
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0,6699.0,7660,8898,8873,...,8295.4,5,2971.0,34.390554,2648.0,39.528288,14.0,7.0,33.0,25.0
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0,4606.0,5371,5834,6166,...,5784.4,5,3201.0,27.462251,2339.0,50.781589,11.0,11.0,9.0,2.0


In [49]:
print("MASTER DATASET VALIDATION")
print("-------------------------")

print("Rows:", master_df.shape[0])
print("Columns:", master_df.shape[1])

print(
    "Unique State/UTs:",
    master_df["state"].nunique()
)

print(
    "Duplicate State/UTs:",
    master_df["state"].duplicated().sum()
)

print(
    "Total missing values:",
    master_df.isnull().sum().sum()
)

print("\nMissing values by column:")
display(
    master_df.isnull().sum()[
        master_df.isnull().sum() > 0
    ]
)

MASTER DATASET VALIDATION
-------------------------
Rows: 36
Columns: 26
Unique State/UTs: 36
Duplicate State/UTs: 0
Total missing values: 8

Missing values by column:


2020_accidents                          1
2020_killed                             1
fatalities_per_100_accidents            1
accident_change_2020_to_2024            1
accident_percent_change_2020_to_2024    1
fatality_change_2020_to_2024            1
fatality_percent_change_2020_to_2024    2
dtype: int64

In [50]:
master_accident_totals = master_df[
    accident_columns
].sum()

print("Accident totals from master dataset:")
display(master_accident_totals)

Accident totals from master dataset:


2020_accidents    372181.0
2021_accidents    412432.0
2022_accidents    461312.0
2023_accidents    480583.0
2024_accidents    487707.0
dtype: float64

In [51]:
master_fatality_totals = master_df[
    fatality_columns
].sum()

print("Fatality totals from master dataset:")
display(master_fatality_totals)

Fatality totals from master dataset:


2020_killed    138383.0
2021_killed    153972.0
2022_killed    168491.0
2023_killed    172890.0
2024_killed    177175.0
dtype: float64

In [52]:
display(
    master_df[
        [
            "state",
            "2024_accidents",
            "2024_killed",
            "population_2024",
            "accidents_per_100k_population",
            "fatalities_per_100k_population",
            "fatalities_per_100_accidents"
        ]
    ].sort_values(
        "2024_killed",
        ascending=False
    ).head(15)
)

,state,2024_accidents,2024_killed,population_2024,accidents_per_100k_population,fatalities_per_100k_population,fatalities_per_100_accidents
26,Uttar Pradesh,46052.0,24118,238078000,19.343240,10.130293,52.371233
22,Tamil Nadu,67526.0,18449,77089000,87.594858,23.932079,27.321328
13,Maharashtra,36118.0,15715,127360000,28.358982,12.339039,43.510161
12,Madhya Pradesh,56669.0,14791,87610000,64.683255,16.882776,26.100690
10,Karnataka,43062.0,12390,68115000,63.219555,18.189826,28.772468
20,Rajasthan,24838.0,11790,81897000,30.328339,14.396132,47.467590
3,Bihar,11610.0,9347,128592000,9.028555,7.268726,80.508183
0,Andhra Pradesh,19557.0,8346,53340000,36.664792,15.646794,42.675257
23,Telangana,25986.0,7949,38272000,67.898202,20.769753,30.589548
6,Gujarat,15588.0,7717,72367000,21.540205,10.663700,49.506030


In [53]:
master_output_path = (
    "../data/processed/"
    "master_state_analysis_2024.csv"
)

master_df.to_csv(
    master_output_path,
    index=False
)

print("Master analytical dataset saved:")
print(master_output_path)

Master analytical dataset saved:
../data/processed/master_state_analysis_2024.csv


In [54]:
verification_df = pd.read_csv(
    master_output_path
)

print("Verification successful.")

print(
    f"Rows: {verification_df.shape[0]}"
)

print(
    f"Columns: {verification_df.shape[1]}"
)

print(
    "Unique State/UTs:",
    verification_df["state"].nunique()
)

display(verification_df.head())

Verification successful.
Rows: 36
Columns: 26
Unique State/UTs: 36


,state,2020_accidents,2021_accidents,2022_accidents,2023_accidents,2024_accidents,2020_killed,2021_killed,2022_killed,2023_killed,...,five_year_average_fatalities,fatality_years_available,accident_change_2020_to_2024,accident_percent_change_2020_to_2024,fatality_change_2020_to_2024,fatality_percent_change_2020_to_2024,accident_rank_2024,fatality_rank_2024,accident_rate_rank_2024,fatality_rate_rank_2024
0,Andhra Pradesh,19509.0,21556.0,21249.0,19949.0,19557.0,7039.0,8186,8293,8137,...,8000.2,5,48.0,0.246040,1307.0,18.567978,9.0,8.0,11.0,8.0
1,Arunachal Pradesh,134.0,283.0,227.0,287.0,277.0,73.0,157,148,145,...,138.2,5,143.0,106.716418,95.0,130.136986,27.0,27.0,24.0,18.0
2,Assam,6595.0,7411.0,7023.0,7421.0,7848.0,2629.0,3036,2994,3296,...,3061.2,5,1253.0,18.999242,722.0,27.462914,16.0,18.0,19.0,22.0
3,Bihar,8639.0,9553.0,10801.0,11014.0,11610.0,6699.0,7660,8898,8873,...,8295.4,5,2971.0,34.390554,2648.0,39.528288,14.0,7.0,33.0,25.0
4,Chhattisgarh,11656.0,12375.0,13279.0,13468.0,14857.0,4606.0,5371,5834,6166,...,5784.4,5,3201.0,27.462251,2339.0,50.781589,11.0,11.0,9.0,2.0


## Master Dataset — Conclusion

The master analytical dataset combines State/UT-level accident, fatality and
population information into a single table.

It provides multiple dimensions of the road-safety problem:

- Absolute accident burden.
- Absolute fatality burden.
- Population-normalized accident burden.
- Population-normalized fatality burden.
- Fatalities per 100 reported accidents.
- Five-year accident and fatality averages.
- Changes between 2020 and 2024.
- 2024 State/UT rankings.

This dataset will serve as the primary analytical input for subsequent
statistical analysis and dashboard development.

The population figures are projected 2024 values rather than a 2024 census
enumeration, and population-normalized metrics therefore represent
descriptive indicators rather than complete measures of road-traffic risk.

Ladakh's unavailable 2020 fatality value is retained as missing and is not
treated as zero.